# Phase 5 — Architecture 3: Legal Prefix–Conv LM

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

SESSION_START = time.time()
ARCH = "A3"
METHODS = ['partial', 'distill']
P1_SOURCE = "/kaggle/input/datasets/abhishekmahti/a3-p1"
HF_REVISION = ""
HOURS = 5.5
OUT = Path("/kaggle/working") / f"phase5_{ARCH}"
OUT.mkdir(parents=True, exist_ok=True)
SRC = OUT / "code"
SRC.mkdir(exist_ok=True)
os.chdir(SRC)
sys.path.insert(0,str(SRC))
import torch
print(ARCH, METHODS, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

A3 ['partial', 'distill'] ['Tesla T4', 'Tesla T4']


## Legal Prefix–Conv LM backbone and data

In [2]:
%%writefile /kaggle/working/phase5_A3/code/backbone.py
import math
# Architecture 3: Legal Prefix–Conv LM — RoPE, RMSNorm, SwiGLU, gated causal convolution
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

def norm(x, w):
    y = x.float()
    return (y * torch.rsqrt(y.square().mean(-1, keepdim=True) + 1e-6) * w).to(x.dtype)

def rope(x, cos, sin):
    a, b = x.float().chunk(2, -1)
    return torch.cat((a*cos-b*sin, b*cos+a*sin), -1).to(x.dtype)

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d, f = c['width'], c['ff']
        self.h = c['heads']
        self.p = nn.ParameterDict({k: nn.Parameter(torch.empty(*shape)) for k, shape in dict(
            n1=(d,), n2=(d,), nc=(d,), conv=(3,d), gate=(d,),
            qkv=(d,3*d), out=(d,d), up=(d,2*f), down=(f,d)).items()})
        for name, w in self.p.items():
            if name in ('n1','n2','nc'): nn.init.ones_(w)
            elif name == 'gate': nn.init.zeros_(w)
            else: nn.init.normal_(w, std=0.02 / math.sqrt(2*c['layers']) if name in ('out','down') else 0.02)

    def forward(self, x, mask, cos, sin, cache=None, pos=0, capacity=0):
        p = self.p
        u = norm(x, p['nc'])
        history = x.new_zeros(x.size(0),2,x.size(-1)) if cache is None else cache[2]
        z = torch.cat((history,u),1)
        local = sum(z[:,2-i:2-i+x.size(1)] * p['conv'][i].to(x.dtype) for i in range(3))
        x = x + torch.sigmoid(u + p['gate'].to(x.dtype)) * local
        q, k, v = (a.view(x.size(0),x.size(1),self.h,-1).transpose(1,2)
                   for a in (norm(x,p['n1']) @ p['qkv']).chunk(3,-1))
        q, k = rope(q,cos,sin), rope(k,cos,sin)
        if cache is not None:
            cache[0][:,:,pos:pos+x.size(1)].copy_(k)
            cache[1][:,:,pos:pos+x.size(1)].copy_(v)
            cache[2].copy_(z[:,-2:])
            k, v = cache[0][:,:,:pos+x.size(1)], cache[1][:,:,:pos+x.size(1)]
        a = F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)
        x = x + a.transpose(1,2).reshape_as(x) @ p['out']
        a, b = (norm(x,p['n2']) @ p['up']).chunk(2,-1)
        x = x + (F.silu(a)*b) @ p['down']
        if capacity:
            keys = k.new_zeros(k.size(0),self.h,capacity,k.size(-1)); values = torch.zeros_like(keys)
            keys[:,:,:k.size(2)].copy_(k); values[:,:,:v.size(2)].copy_(v)
            return x, (keys,values,z[:,-2:].clone())
        return x

class Model(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.embed = nn.Parameter(torch.empty(c['vocab'],c['width']))
        nn.init.normal_(self.embed,std=0.02)
        self.final = nn.Parameter(torch.ones(c['width']))
        self.blocks = nn.ModuleList([Block(c) for _ in range(c['layers'])])
        half = c['width']//c['heads']//2
        angle = torch.arange(c['prefix']+1024).float()[:,None] * 10000.**(-torch.arange(half).float()/half)
        self.register_buffer('cos', angle.cos()[None,None], persistent=False)
        self.register_buffer('sin', angle.sin()[None,None], persistent=False)

    def hidden(self, ids, prefix):
        t = ids.size(1)
        idx = torch.arange(t,device=ids.device)
        q, k, pre = idx[None,:,None], idx[None,None,:], prefix[:,None,None]
        mask = (((q<pre)&(k<pre)) | ((q>=pre)&(k<=q)))[:,None]
        x = F.embedding(ids,self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block in self.blocks:
            args = (x,mask,self.cos[:,:,:t],self.sin[:,:,:t])
            x = checkpoint(block,*args,use_reentrant=False) if self.training else block(*args)
        return norm(x,self.final)

    def forward(self, rows):
        c = self.c
        ids, prefix, length = rows[:,:c['seq']], rows[:,-2], rows[:,-1]
        x = self.hidden(ids,prefix)
        offsets = torch.arange(c['target'],device=rows.device)[None]
        idx = prefix[:,None]-1+offsets
        states = x.gather(1,idx[:,:,None].expand(-1,-1,x.size(-1)))
        labels = ids.gather(1,idx+1)
        logits = states @ self.embed.T
        loss = F.cross_entropy(logits.float().flatten(0,1),labels.flatten(),reduction='none').view_as(labels)
        return ((loss*(offsets<length[:,None])).sum(1)/length).mean()

    def prefill(self, ids):
        t = ids.size(1); cache = []
        x = F.embedding(ids,self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block in self.blocks:
            x, kv = block(x,None,self.cos[:,:,:t],self.sin[:,:,:t],capacity=t+1024)
            cache.append(kv)
        return (norm(x[:,-1],self.final) @ self.embed.T).float(), cache

    def decode(self, token, cache, pos):
        x = F.embedding(token[:,None],self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block, kv in zip(self.blocks,cache):
            x = block(x,None,self.cos[:,:,pos:pos+1],self.sin[:,:,pos:pos+1],kv,pos)
        return (norm(x[:,0],self.final) @ self.embed.T).float()

Writing /kaggle/working/phase5_A3/code/backbone.py


In [3]:
%%writefile /kaggle/working/phase5_A3/code/p5_data.py
import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np

def write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    os.replace(tmp, path)

def digest(path):
    with Path(path).open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else stream_digest(f)

def stream_digest(f):
    h = hashlib.sha256()
    for chunk in iter(lambda: f.read(4 << 20), b""):
        h.update(chunk)
    return h.hexdigest()

def config_matches(c, arch):
    signatures = {"A1": "ffn_dim" in c and "layers" in c and "encoder_layers" not in c,
                  "A2": "encoder_layers" in c and "decoder_layers" in c,
                  "A3": "prefix" in c and "width" in c,
                  "A4": "kv_heads" in c and "global_every" in c}
    return signatures[arch]

def foundation(location, arch, out):
    def compatible(raw):
        return config_matches(json.loads(raw), arch)
    def inspect_path(path):
        if path.is_dir():
            config = path / "config.json"
            if config.is_file() and (path / "best.pt").is_file() and (path / "tokenizer.json").is_file() and compatible(config.read_bytes()):
                return (path, None)
        elif path.is_file() and path.suffix.lower() == ".zip":
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    if Path(name).name != "config.json":
                        continue
                    prefix = name[:-len("config.json")]
                    if {prefix + k for k in ("best.pt", "tokenizer.json")} <= set(z.namelist()) and compatible(z.read(name)):
                        return (path, prefix)
        return None
    if location:
        candidates = [Path(location)]
    else:
        root = Path("/kaggle/input")
        candidates = sorted({p.parent for p in root.rglob("best.pt")}) 
        candidates += sorted(p for p in root.rglob("*.zip") if "bundle" in p.name.lower())
    matches = []
    for path in candidates:
        found = inspect_path(path)
        if found:
            matches.append(found)
    if not location and any(prefix is None for _, prefix in matches):
        matches = [(p, prefix) for p, prefix in matches if prefix is None]
    if len(matches) != 1:
        raise ValueError(f"Set P1_SOURCE to exactly one {arch} Phase-1 output directory or bundle ZIP. Matching paths: {matches}")
    path, prefix = matches[0]
    if prefix is None:
        return path
    dest = out / "foundation"
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as z:
        for name in ("config.json", "tokenizer.json", "best.pt"):
            with z.open(prefix + name) as src, (dest / name).open("wb") as target:
                shutil.copyfileobj(src, target)
    return dest

def split_paths(revision):
    rel = {"train":"Train/train.json", "dev":"Train/dev.json", "test":"Test/test.json"}
    from huggingface_hub import HfApi, hf_hub_download
    token = os.environ["HF_TOKEN"]
    repo = "Exploration-Lab/CS-781-Capstone"
    commit = HfApi(token=token).dataset_info(repo, revision=revision or "main").sha
    paths = {k: Path(hf_hub_download(repo, "Phase_5_LSI/" + v, repo_type="dataset",
                                     revision=commit, token=token)) for k, v in rel.items()}
    return paths, f"{repo}@{commit}"

def read_cases(path, split):
    raw = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    if isinstance(raw, dict) and isinstance(raw.get("data"), list):
        raw = raw["data"]
    if isinstance(raw, dict):
        rows = []
        for key, val in raw.items():
            rec = {"text": val} if isinstance(val, str) else dict(val)
            if "id" in rec and str(rec["id"]) != key:
                raise ValueError(f"{split}: conflicting id {key}")
            rec.setdefault("id", key)
            rows.append(rec)
    else:
        rows = raw
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"Empty or unsupported {split} file: {path}")
    result, seen = [], set()
    for r in rows:
        key, body = r["id"], r["text"]
        if type(key) not in (int, str) or str(key) in seen or not isinstance(body, str):
            raise ValueError(f"{split}: invalid case or duplicate id {key!r}")
        seen.add(str(key))
        labels = []
        if split != "test":
            vals = r["labels"]
            if not isinstance(vals, list) or any(type(x) is not int and not (isinstance(x, str) and x.strip().lstrip("-").isdigit()) for x in vals):
                raise ValueError(f"{split}: invalid statute list for {key}")
            labels = sorted(set(map(int, vals)))
        result.append((key, body, labels))
    return result

def starts(n, width, stride):
    last = max(0, n - width)
    return list(range(0, last, stride)) + [last]

def cache_data(c, paths, source):
    import re
    from tokenizers import Tokenizer
    out = Path(c["out"])
    tok = Tokenizer.from_file(str(Path(c["base"]) / "tokenizer.json"))
    tok.no_truncation()
    tok.no_padding()
    pad = tok.token_to_id("<pad>")
    if pad is None:
        raise ValueError("The Phase-1 tokenizer has no <pad> token.")
    cues = re.compile(r"\b(?:sections?|articles?|act|code|statute|rules?|ordinance|punishable|provisions?|charged|held)\b|\b\d+[A-Z]?\s*\(", re.I)
    vocab = set()
    meta = {"pad": pad, "tokenizer_size": tok.get_vocab_size(), "source": source, "files": {}}
    for split, path in paths.items():
        cases = read_cases(path, split)
        ids, labels, offsets, evidence = [], [], [0], []
        with (out / f"{split}.i32").open("wb") as f:
            for key, body, labs in cases:
                enc = tok.encode(body, add_special_tokens=False)
                tokens = np.asarray(enc.ids or [pad], dtype=np.int32)
                tokens.tofile(f)
                offsets.append(offsets[-1] + len(tokens))
                ids.append(key)
                labels.append(labs)
                if split != "test":
                    vocab.update(labs)
                windows = starts(len(tokens), c["width"], c["stride"])
                cue_pos = np.asarray([m.start() for m in cues.finditer(body)], dtype=np.int64)
                bounds = enc.offsets
                scores = []
                for s in windows:
                    lo = bounds[s][0] if bounds else 0
                    hi = bounds[min(s+c["width"], len(tokens))-1][1] if bounds else 0
                    scores.append(int(np.searchsorted(cue_pos, hi) - np.searchsorted(cue_pos, lo)))
                evidence.append(scores)
        write_json(out / f"{split}.json", dict(ids=ids, labels=labels, offsets=offsets, evidence=evidence))
        meta["files"][split] = {"sha256": digest(path), "cases": len(ids), "tokens": offsets[-1],
                                "median_tokens": float(np.median(np.diff(offsets))),
                                "max_tokens": int(max(np.diff(offsets)))}
    if not vocab:
        raise ValueError("No statute IDs in the labelled splits.")
    meta["labels"] = sorted(vocab)
    meta["tokenizer_sha256"] = digest(Path(c["base"]) / "tokenizer.json")
    meta["foundation_sha256"] = digest(Path(c["base"]) / "best.pt")
    write_json(out / "data_manifest.json", meta)
    return meta

class Cases:
    def __init__(self, c, split):
        out = Path(c["out"])
        self.info = json.loads((out / f"{split}.json").read_text())
        meta = json.loads((out / "data_manifest.json").read_text())
        self.pad, self.labels, self.ids = meta["pad"], meta["labels"], self.info["ids"]
        self.tokens = np.memmap(out / f"{split}.i32", dtype=np.int32, mode="r")
        self.width, self.stride = c["width"], c["stride"]
        self.y = np.zeros((len(self.ids), len(self.labels)), dtype=np.float32)
        col = {lab: i for i, lab in enumerate(self.labels)}
        for i, labs in enumerate(self.info["labels"]):
            self.y[i, [col[x] for x in labs]] = 1

    def windows(self, i, count, mode="selected", rng=None, length=None):
        a, b = self.info["offsets"][i:i+2]
        width = length or self.width
        pos = starts(b-a, width, min(self.stride, max(1,width*3//4)))
        if length and length != self.width:
            old = starts(b-a, self.width, self.stride)
            scores = np.interp(pos, old, self.info["evidence"][i])
        else:
            scores = np.asarray(self.info["evidence"][i])
        if mode == "all" or len(pos) <= count:
            keep = range(len(pos))
        elif mode == "uniform":
            keep = np.linspace(0, len(pos)-1, count).round().astype(int)
        elif mode == "opening":
            keep = range(count)
        else:
            ranked = sorted(range(len(pos)), key=lambda j: (-scores[j],j))
            order = [ranked[0], len(pos)-1, 0] + ranked[1:2] + list(np.linspace(0,len(pos)-1,count+2).round().astype(int)) + ranked
            keep = list(dict.fromkeys(order))[:count]
            if rng is not None and count >= 2 and rng.random() < .5:
                others = sorted(set(range(len(pos)))-set(keep))
                if others:
                    keep[-1] = int(rng.choice(others))
        return [self.tokens[a+pos[j]:min(b,a+pos[j]+width)] for j in sorted(keep)]

Writing /kaggle/working/phase5_A3/code/p5_data.py


## Multi-label metrics and calibration

In [4]:
%%writefile /kaggle/working/phase5_A3/code/p5_metrics.py
import hashlib
import json
import os
import zipfile
from pathlib import Path
import numpy as np

def development_parts(ids, seed=781):
    if len(ids) < 2:
        raise ValueError("Dev needs at least two distinct cases for calibration and held-out scoring.")
    order = sorted(range(len(ids)), key=lambda j: hashlib.sha256(f"{seed}|{type(ids[j]).__name__}|{ids[j]}".encode()).digest())
    return np.asarray(order[::2]), np.asarray(order[1::2])

def decide(probs, thresholds, seen):
    if not np.isfinite(probs).all():
        raise FloatingPointError("Nonfinite probability")
    masked = np.where(seen, probs, -1.)
    result = (masked >= thresholds) & seen
    empty = ~result.any(1)
    result[empty, masked[empty].argmax(1)] = True
    return result

def metrics(y, pred):
    truth = y.astype(bool)
    tp = np.sum(truth & pred, axis=0)
    fp = pred.sum(0) - tp
    fn = truth.sum(0) - tp
    p = tp/np.maximum(1,tp+fp)
    r = tp/np.maximum(1,tp+fn)
    f = 2*tp/np.maximum(1,2*tp+fp+fn)
    return dict(macro_f1=float(f.mean()), micro_f1=float(2*tp.sum()/max(1,(2*tp+fp+fn).sum())),
                subset_accuracy=float((truth == pred).all(1).mean()), macro_precision=float(p.mean()),
                macro_recall=float(r.mean()), mean_predictions=float(pred.sum(1).mean()))

def calibrate(p, y, seen):
    grid = np.unique(np.r_[.002,.005,.01,.02,np.linspace(.03,.51,17), .6, .7, .8, .9])
    macro = [metrics(y,decide(p,t,seen))["macro_f1"] for t in grid]
    global_t = float(grid[np.argmax(macro)])
    arr = np.full(p.shape[1], global_t)
    for j in np.where((y.sum(0)>=3) & seen)[0]:
        yt = y[:,j].astype(bool)
        guesses = p[:,j,None] >= grid
        f = 2*(guesses & yt[:,None]).sum(0) / np.maximum(1, guesses.sum(0)+yt.sum())
        best = np.flatnonzero(f == f.max())
        t = grid[best[np.argmin(np.abs(grid[best]-global_t))]]
        blend = yt.sum()/(yt.sum()+10)
        arr[j] = global_t*(1-blend) + t*blend
    score_local = metrics(y, decide(p, arr, seen))["macro_f1"]
    policy = "shrunk_per_label" if score_local > max(macro) else "global"
    return (arr if policy == "shrunk_per_label" else np.full_like(arr,global_t)), global_t, policy

def diagnostics(y, pred, training, labels):
    truth = y.astype(bool)
    tp = (truth & pred).sum(0)
    fn = (truth & ~pred).sum(0)
    fp = (~truth & pred).sum(0)
    per_label = [dict(label=label, train=int(training[j]), support=int(truth[:,j].sum()),
                      tp=int(tp[j]), fp=int(fp[j]), fn=int(fn[j]),
                      f1=float(2*tp[j]/max(1,2*tp[j]+fp[j]+fn[j]))) for j,label in enumerate(labels)]
    groups = {}
    for name, mask in {"unseen":training==0, "rare_1_to_10":(training>0)&(training<=10),
                       "over_10":training>10}.items():
        if mask.any():
            groups[name] = metrics(y[:,mask],pred[:,mask])
    missed_extra = (~pred & truth).astype(np.int64).T @ (pred & ~truth).astype(np.int64)
    co = truth.astype(np.int64).T @ truth.astype(np.int64)
    def top(mat):
        pairs = sorted(zip(*np.nonzero(mat)), key=lambda ij:-int(mat[ij]))[:20]
        return [[labels[i],labels[j],int(mat[i,j])] for i,j in pairs]
    return {"by_statute":per_label,"by_train_frequency":groups,"missed_with_extra":top(missed_extra),
            "true_cooccurrence":top(np.triu(co,1))}

def validate(rows, ids, labels):
    if len(rows) != len(ids) or len({str(r["id"]) for r in rows}) != len(ids):
        raise ValueError("Missing or duplicate test IDs")
    for r, expected in zip(rows,ids):
        p = r["pred"]
        if set(r) != {"id","pred"} or type(r["id"]) is not type(expected) or r["id"] != expected:
            raise ValueError(f"ID mismatch: {expected!r}")
        if not isinstance(p,list) or not p or any(type(x) is not int or x not in labels for x in p) or len(set(p))!=len(p):
            raise ValueError(f"Invalid statutes for {expected!r}")

def export(folder, ids, labels, binary):
    rows = [{"id":i, "pred":[labels[j] for j in np.flatnonzero(row)]} for i,row in zip(ids,binary)]
    validate(rows,ids,set(labels))
    data = json.dumps(rows, ensure_ascii=False,separators=(",",":")).encode()
    folder = Path(folder)
    (folder/"prediction.json").write_bytes(data)
    tmp = folder/"submission.tmp"
    with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json",data)
    with zipfile.ZipFile(tmp) as z:
        assert z.namelist() == ["prediction.json"]
        validate(json.loads(z.read("prediction.json")),ids,set(labels))
    os.replace(tmp,folder/"submission.zip")
    return hashlib.sha256(data).hexdigest()

Writing /kaggle/working/phase5_A3/code/p5_metrics.py


## Document classifier

In [5]:
%%writefile /kaggle/working/phase5_A3/code/p5_model.py
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
import backbone
from p5_data import config_matches

class MultiplicativeProjection(nn.Module):
    """LoRMA+ applies (I + alpha/r BA) to the output of the frozen projection."""
    def __init__(self, projection, rank=8, alpha=16):
        super().__init__()
        self.base = projection
        size = projection.out_features
        self.a = nn.Linear(size, rank, bias=False)
        self.b = nn.Linear(rank, size, bias=False)
        nn.init.zeros_(self.b.weight)
        self.scale = alpha/rank

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * self.b(self.a(y))

class StatuteModel(nn.Module):
    def __init__(self, base, cfg, arch, method, labels):
        super().__init__()
        self.base, self.arch = base, arch
        blocks = base.encoder if arch == "A2" else base.blocks
        if method == "distill":
            base.blocks = nn.ModuleList(list(blocks)[1::2])
        if method in {"partial", "lorma"}:
            base.requires_grad_(False)
        if method == "partial":
            for layer in blocks[-max(1,len(blocks)//3):]:
                layer.requires_grad_(True)
            if arch == "A2":
                for layer in base.decoder[-max(1,len(base.decoder)//3):]:
                    layer.requires_grad_(True)
                base.encoder_norm.requires_grad_(True)
                base.decoder_norm.requires_grad_(True)
            elif arch == "A3":
                base.final.requires_grad_(True)
            else:
                base.norm.requires_grad_(True)
        if method == "lorma":
            for block in blocks:
                block.attn.qkv = MultiplicativeProjection(block.attn.qkv)
                block.attn.out = MultiplicativeProjection(block.attn.out)
        d = cfg.get("dim",cfg.get("width"))
        self.head = nn.Sequential(nn.LayerNorm(2*d),nn.Linear(2*d,d//2),nn.GELU(),nn.Dropout(.15))
        self.score = nn.Linear(d//2,labels)
        self.gate = nn.Linear(d//2,labels)
        nn.init.zeros_(self.gate.weight)
        nn.init.zeros_(self.gate.bias)

    def chunk_features(self, ids, mask):
        b = self.base
        lengths = mask.sum(1)
        if self.arch == "A1":
            x = b.embed(ids)
            pos = torch.arange(ids.size(1),device=ids.device)[None,:]
            for layer in b.blocks:
                x = layer(x,pos)[0]
            x = b.norm(x)
        elif self.arch == "A2":
            x = b.encode(ids,mask)
            start = torch.full_like(ids[:,:1],b.cfg["bos_id"])
            last = b.decode(start,x,mask)[0][:,0]
        elif self.arch == "A3":
            x = b.hidden(ids,lengths)
        else:
            x = b.emb(ids)
            for layer in b.blocks:
                x = layer(x)[0]
            x = b.norm(x)
        if self.arch != "A2":
            last = x[torch.arange(len(x),device=x.device), lengths-1]
        pooled = (x.float()*mask.unsqueeze(-1)).sum(1)/lengths[:,None]
        return self.head(torch.cat([pooled,last.float()],dim=1))

    def forward(self, docs, pad, mode="attention", pack=8):
        device = next(self.parameters()).device
        windows = [w for doc in docs for w in doc]
        feats = []
        for i in range(0,len(windows),pack):
            part = windows[i:i+pack]
            width = max(len(w) for w in part)
            a = np.full((len(part),width),pad,dtype=np.int64)
            for j,w in enumerate(part):
                a[j,:len(w)] = w
            ids = torch.as_tensor(a,device=device)
            sizes = torch.tensor([len(w) for w in part],device=device)
            mask = torch.arange(width,device=device)[None,:] < sizes[:,None]
            feats.append(self.chunk_features(ids,mask))
        feats = torch.cat(feats)
        value, weight = self.score(feats).float(), self.gate(feats).float()
        counts = [len(doc) for doc in docs]
        out = []
        for logits, gates in zip(value.split(counts),weight.split(counts)):
            out.append(logits.mean(0) if mode=="mean" else (logits*gates.softmax(0)).sum(0))
        return torch.stack(out)

def make_model(c, method, meta):
    bundle = torch.load(Path(c["base"])/"best.pt",map_location="cpu",weights_only=True,mmap=True)
    cfg = bundle["config"]
    if not config_matches(cfg,c["arch"]):
        raise ValueError("Checkpoint architecture does not match notebook.")
    if bundle.get("tokenizer_sha256",meta["tokenizer_sha256"]) != meta["tokenizer_sha256"]:
        raise ValueError("Checkpoint and tokenizer fingerprints differ.")
    v = cfg.get("vocab_size",cfg.get("vocab"))
    if meta["tokenizer_size"] > v:
        raise ValueError("Tokenizer exceeds checkpoint vocabulary.")
    cls = getattr(backbone,{"A1":"LegalLM","A2":"LegalSeq2Seq","A3":"Model","A4":"LegalLM"}[c["arch"]])
    base = cls(cfg)
    base.load_state_dict(bundle["model"], strict=True)
    return StatuteModel(base,cfg,c["arch"],method,len(meta["labels"]))

def trainable_state(model):
    return {n:p.detach().cpu() for n,p in model.named_parameters() if p.requires_grad}

def restore_weights(model,state):
    names = {n for n,p in model.named_parameters() if p.requires_grad}
    if set(state) != names:
        raise ValueError("Saved adaptation does not match model")
    model.load_state_dict(state,strict=False)

Writing /kaggle/working/phase5_A3/code/p5_model.py


## Training and checkpoint selection

In [6]:
%%writefile /kaggle/working/phase5_A3/code/p5_train.py
import argparse
import hashlib
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import torch
from torch.nn import functional as F
from p5_data import Cases, write_json, digest
from p5_metrics import calibrate, decide, development_parts, diagnostics, export, metrics
from p5_model import make_model, restore_weights, trainable_state

def save_pt(path, payload):
    tmp = path.with_suffix(".tmp")
    torch.save(payload,tmp)
    os.replace(tmp,path)

def amp(device):
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=="cuda")

@torch.inference_mode()
def predict(model, cases, indices, c, device, deadline, count=None, selection="selected", pool="attention", length=None):
    model.eval()
    result = []
    for start in range(0,len(indices),c["eval_batch"]):
        if time.time() >= deadline:
            raise TimeoutError("Inference deadline reached before covering every requested case")
        chunk = indices[start:start+c["eval_batch"]]
        docs = [cases.windows(i,count or c["eval_windows"],selection,length=length) for i in chunk]
        with amp(device):
            logits = model(docs,cases.pad,pool)
        result.append(logits.sigmoid().cpu().numpy())
    return np.concatenate(result)

def run(c, method, gpu, deadline):
    device = torch.device(gpu)
    if device.type == "cuda":
        torch.cuda.set_device(device)
    torch.set_num_threads(2)
    torch.manual_seed(c["seed"])
    rng = np.random.default_rng(c["seed"])
    root = Path(c["run_dir"])
    folder = root/method
    folder.mkdir(exist_ok=True)
    meta = json.loads((Path(c["out"])/"data_manifest.json").read_text())
    signature_data = {k:v for k,v in c.items() if k not in {"out","base","run_dir"}}
    signature_data.update(method=method, files=meta["files"], tokenizer=meta["tokenizer_sha256"],
                          base_hash=meta["foundation_sha256"], labels=meta["labels"])
    if method=="distill":
        signature_data["teacher_hash"] = digest(root/"partial"/"best.pt")
    signature = hashlib.sha256(json.dumps(signature_data,sort_keys=True).encode()).hexdigest()
    started = time.time()
    train, dev, test = (Cases(c,name) for name in ("train","dev","test"))
    support = train.y.sum(0)
    seen = support>0
    if not seen.any():
        raise ValueError("Train split has no positive statutes.")
    calib, hold = development_parts(dev.ids,c["seed"])
    monitor = calib[:min(192,len(calib))]
    model = make_model(c,method,meta).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    weights = np.clip(np.sqrt((len(train.ids)-support)/np.maximum(1,support)),1,8)
    if c["loss"] in {"bce", "asymmetric"}:
        weights = np.ones_like(weights)
    pos = torch.tensor(weights, device=device)
    prior = (support+1)/(len(train.ids)+2)
    with torch.no_grad():
        model.score.bias.copy_(torch.tensor(np.log(prior/(1-prior)*weights),device=device))
    teacher = None
    if method=="distill":
        td = torch.device("cuda:0") if device.type=="cuda" else device
        teacher = make_model(c,"partial",meta)
        tbest = torch.load(root/"partial"/"best.pt",map_location="cpu",weights_only=True)
        restore_weights(teacher,tbest["weights"])
        del tbest
        teacher = teacher.to(td).eval().requires_grad_(False)
        assert sum(p.numel() for p in model.parameters()) < sum(p.numel() for p in teacher.parameters())
    lr = {"full":2e-5,"partial":4e-5,"lorma":3e-4,"distill":4e-5}[method]
    body = [p for p in model.base.parameters() if p.requires_grad]
    head = [p for n,p in model.named_parameters() if not n.startswith("base.")]
    opt = torch.optim.AdamW([{"params":body,"lr":lr},{"params":head,"lr":5e-4}],weight_decay=.02)
    scaler = torch.amp.GradScaler("cuda",enabled=device.type=="cuda")
    step, best, examples_seen = 0,-1.,0
    order = rng.permutation(len(train.ids))
    cursor = 0
    eval_record = []
    probe = monitor[:min(32,len(monitor))]
    t = time.time()
    predict(model,dev,probe,c,device,deadline)
    seconds_per_case = (time.time()-t)/len(probe)
    remaining = deadline-time.time()
    nfinal = len(dev.ids)+len(test.ids)
    chosen_windows = c["eval_windows"]
    if seconds_per_case*nfinal*1.5 > remaining*.45:
        chosen_windows = min(2,chosen_windows)
    if seconds_per_case*nfinal*1.5*chosen_windows/c["eval_windows"] > remaining*.60:
        chosen_windows = 1
    reserve = max(c["reserve_minutes"]*60, seconds_per_case*nfinal*1.6*chosen_windows/c["eval_windows"]+120)
    stop = deadline - reserve
    print(f"{method}: {len(train.ids)} train, {nfinal} final cases, {seconds_per_case:.3f}s/case, "
          f"{chosen_windows} final windows, reserve {reserve/60:.1f} min",flush=True)
    effective = c["micro"]*c["accum"]
    epoch_steps = max(1,math.ceil(len(train.ids)/effective))
    max_steps = epoch_steps*c["epochs"]
    eval_every = max(8,min(160,epoch_steps//2 or 1))
    last_eval = -1
    def evaluate():
        nonlocal best,last_eval
        t = time.time()
        p = predict(model,dev,monitor,c,device,deadline,count=chosen_windows)
        threshold,_,_ = calibrate(p,dev.y[monitor],seen)
        score = metrics(dev.y[monitor],decide(p,threshold,seen))["macro_f1"]
        eval_record.append(dict(step=step,monitor_macro_f1=score,elapsed_seconds=round(time.time()-started,2)))
        if score > best:
            best = score
            save_pt(folder/"best.pt",dict(weights=trainable_state(model),score=score,step=step,signature=signature))
        print(f"{method} step {step}, calibration subset macro-F1 {score:.4f}, eval {time.time()-t:.1f}s",flush=True)
        last_eval = step
    while step < max_steps and (time.time()<stop or step==0):
        model.train()
        opt.zero_grad(set_to_none=True)
        warmup = max(1,min(32,epoch_steps//4))
        factor = min(1.,(step+1)/warmup)*(.3+.7*.5*(1+math.cos(math.pi*min(1,step/max_steps))))
        opt.param_groups[0]["lr"],opt.param_groups[1]["lr"] = lr*factor,5e-4*factor
        for _ in range(c["accum"]):
            if cursor>=len(order):
                order,cursor = rng.permutation(len(order)),0
            ix = order[cursor:cursor+c["micro"]]
            cursor += len(ix)
            examples_seen += len(ix)
            docs = [train.windows(i,c["train_windows"],rng=rng) for i in ix]
            target = torch.tensor(train.y[ix],device=device)
            if teacher is not None:
                with torch.no_grad(), amp(td):
                    soft = (teacher(docs,train.pad)/c["temperature"]).sigmoid().to(device)
            with amp(device):
                logits = model(docs,train.pad)
                loss = F.binary_cross_entropy_with_logits(logits,target,pos_weight=pos)
                if c["loss"] == "asymmetric":
                    negatives = F.softplus(logits) * logits.sigmoid().detach().square()
                    loss = (target*F.softplus(-logits) + (1-target)*negatives).mean()
                if teacher is not None:
                    T = c["temperature"]
                    kd = F.binary_cross_entropy_with_logits(logits/T,soft)*T*T
                    loss = .55*loss+.45*kd
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite {method} loss at step {step}")
            scaler.scale(loss/c["accum"]).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params,1.)
        before = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        if scaler.get_scale()<before:
            if scaler.get_scale()<1e-5:
                raise FloatingPointError("Repeated FP16 overflow, no stable update.")
            continue
        step += 1
        if step==1 or step%25==0:
            print(f"{method} step {step}/{max_steps}, loss {loss.item():.4f}",flush=True)
        if step == min(16,eval_every) or step%eval_every==0:
            evaluate()
    if last_eval != step:
        evaluate()
    best_ck = torch.load(folder/"best.pt",map_location="cpu",weights_only=True)
    restore_weights(model,best_ck["weights"])
    best_step = best_ck["step"]
    train_peak = torch.cuda.max_memory_allocated(device)/2**30 if device.type=="cuda" else None
    teacher_device_peak = (torch.cuda.max_memory_allocated(td)/2**30 if teacher is not None
                           and device.type=="cuda" and td != device else None)
    del best_ck, opt, teacher
    model.zero_grad(set_to_none=True)
    if device.type=="cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    training_seconds = time.time() - started
    t = time.time()
    p = predict(model,dev,np.arange(len(dev.ids)),c,device,deadline,count=chosen_windows)
    dev_forward_seconds = time.time()-t
    threshold,global_t,policy = calibrate(p[calib],dev.y[calib],seen)
    evaluated = decide(p[hold],threshold,seen)
    baseline_p = np.broadcast_to(prior,dev.y.shape)
    baseline_t,_,_ = calibrate(baseline_p[calib],dev.y[calib],seen)
    test_start = time.time()
    test_p = predict(model,test,np.arange(len(test.ids)),c,device,deadline,count=chosen_windows)
    forward_seconds = dev_forward_seconds + time.time()-test_start
    prediction_hash = export(folder,test.ids,train.labels,decide(test_p,threshold,seen))
    inference_time = time.time()-t
    np.savez_compressed(folder/"dev_probabilities.npz",probabilities=p,calibration=calib,audit=hold,thresholds=threshold,
                        labels=train.labels)
    report = dict(phase=5,architecture=c["arch"],method=method,signature=signature,foundation_sha256=meta["foundation_sha256"],
                  dataset=meta["source"],labels=train.labels,train_cases=len(train.ids),development_cases=len(dev.ids),
                  calibration_cases=len(calib),audit_cases=len(hold),optimizer="AdamW",backbone_lr=lr,head_lr=5e-4,
                  schedule="linear warmup, cosine decay",config=c,
                  best_step=best_step,steps=step,trained_document_passes=examples_seen/len(train.ids),monitor_history=eval_record,
                  calibration_policy=policy,global_threshold=global_t,thresholds=threshold.tolist(),
                  metrics=metrics(dev.y[hold],evaluated),global_metrics=metrics(dev.y[hold],decide(p[hold],global_t,seen)),
                  fixed_half_metrics=metrics(dev.y[hold],decide(p[hold],.5,seen)),frequency_baseline=metrics(dev.y[hold],decide(baseline_p[hold],baseline_t,seen)),
                  diagnostics=diagnostics(dev.y[hold],evaluated,support,train.labels),
                  total_params=sum(x.numel() for x in model.parameters()),trainable_params=sum(x.numel() for x in params),
                  peak_allocated_GiB=train_peak,additional_teacher_gpu_peak_GiB=teacher_device_peak,
                  inference_peak_GiB=torch.cuda.max_memory_allocated(device)/2**30 if device.type=="cuda" else None,
                  gpu=torch.cuda.get_device_name(device) if device.type=="cuda" else "cpu",
                  backbone_file_bytes=(Path(c["base"])/"best.pt").stat().st_size,
                  adaptation_file_bytes=(folder/"best.pt").stat().st_size,
                  deployed_weight_MiB_fp32=sum(x.numel()*4 for x in model.parameters())/2**20,
                  training_and_monitor_seconds=training_seconds,
                  eval_windows=chosen_windows,inference_seconds=inference_time,
                  inference_seconds_per_case=forward_seconds/nfinal,elapsed_seconds=time.time()-started,
                  prediction_sha256=prediction_hash,codabench_id=None,codabench_score=None)
    report["error_examples"] = []
    for local in np.flatnonzero((dev.y[hold].astype(bool) != evaluated).any(1))[:12]:
        i = hold[local]
        missing = np.flatnonzero(dev.y[i].astype(bool) & ~evaluated[local])
        extra = np.flatnonzero(~dev.y[i].astype(bool) & evaluated[local])
        report["error_examples"].append(dict(id=dev.ids[i],tokens=dev.info["offsets"][i+1]-dev.info["offsets"][i],
                    missed=[train.labels[j] for j in missing],extra=[train.labels[j] for j in extra]))
    write_json(folder/"report.json",report)
    print(method,"held-out metrics",report["metrics"],flush=True)
    subset = hold[:min(24,len(hold))]
    comparisons = {"selected_hierarchical":metrics(dev.y[subset],decide(p[subset],threshold,seen))}
    for name,selection,pool,width in [("uniform","uniform","attention",None),("opening_sections","opening","attention",None),
            ("selected_mean","selected","mean",None),("short_context","selected","attention",c["width"]//2),
            ("full_document","all","attention",None)]:
        if time.time()+15 >= deadline:
            break
        probs = predict(model,dev,subset,c,device,deadline-10,count=chosen_windows,selection=selection,pool=pool,length=width)
        comparisons[name] = metrics(dev.y[subset],decide(probs,threshold,seen))
        write_json(folder/"context_ablation.json",dict(case_ids=[dev.ids[i] for i in subset],metrics=comparisons))
    return report

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("config")
    p.add_argument("method")
    p.add_argument("device")
    p.add_argument("deadline",type=float)
    args = p.parse_args()
    run(json.loads(Path(args.config).read_text()),args.method,args.device,args.deadline)

Writing /kaggle/working/phase5_A3/code/p5_train.py


## Model checks

In [7]:
import importlib
import numpy as np
import torch
import backbone, p5_model
importlib.reload(backbone)
importlib.reload(p5_model)
from p5_model import StatuteModel, MultiplicativeProjection
torch.manual_seed(781)
tiny_cfg = dict(dim=32,width=32,vocab=128,vocab_size=128,heads=4,kv_heads=2,layers=4,
                encoder_layers=2,decoder_layers=2,ffn_dim=64,ff=64,window=8,
                global_every=2,prefix=32,gradient_checkpointing=False,bos_id=1,pad_id=0)
base_class = getattr(backbone,{"A1":"LegalLM","A2":"LegalSeq2Seq","A3":"Model","A4":"LegalLM"}[ARCH])
doc = [[np.asarray([2,3,4]), np.asarray([4,5])],[np.asarray([6,7,8,9])]]
for method in METHODS:
    small = StatuteModel(base_class(tiny_cfg), tiny_cfg, ARCH, method, 3).cuda()
    small.eval()
    with torch.no_grad():
        joined = small(doc,0)
        separate = torch.cat([small([x],0) for x in doc])
        torch.testing.assert_close(joined,separate,atol=2e-4,rtol=2e-4)
        for adapter in (m for m in small.modules() if isinstance(m,MultiplicativeProjection)):
            x = torch.randn(2,adapter.base.in_features,device="cuda")
            torch.testing.assert_close(adapter(x),adapter.base(x))
    small.train()
    loss = torch.nn.functional.binary_cross_entropy_with_logits(small(doc,0),torch.tensor([[1.,0.,1.],[0.,1.,0.]],device="cuda"))
    loss.backward()
    assert torch.isfinite(loss) and small.score.weight.grad is not None
    assert all(torch.isfinite(p.grad).all() for p in small.parameters() if p.grad is not None)
    if method=="lorma":
        assert any(m.b.weight.grad is not None and m.b.weight.grad.abs().sum()>0 for m in small.modules() if isinstance(m,MultiplicativeProjection))
    if method=="distill":
        assert len(small.base.blocks)==tiny_cfg["layers"]//2
    print(method, "passed padding, multi-label forward and gradient checks")
    del small, loss
torch.cuda.empty_cache()

partial passed padding, multi-label forward and gradient checks
distill passed padding, multi-label forward and gradient checks


## Prepare data

In [8]:
import importlib
import p5_data
importlib.reload(p5_data)
from p5_data import foundation, split_paths, cache_data, write_json

base = foundation(P1_SOURCE,ARCH,OUT)
config = dict(arch=ARCH,base=str(base),out=str(OUT),width=384,stride=288,
              train_windows=2,eval_windows=4,eval_batch=2,micro=2,accum=4,
              epochs=8,seed=781,temperature=2.0,loss='weighted_bce',reserve_minutes=18)
paths, source = split_paths(HF_REVISION)
manifest = cache_data(config,paths,source)
print("Phase-1 foundation:",base)
print("Data:",source,"| statute IDs:",len(manifest["labels"]))
for split,item in manifest["files"].items():
    print(split,item)

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Phase-1 foundation: /kaggle/input/datasets/abhishekmahti/a3-p1
Data: Exploration-Lab/CS-781-Capstone@93b0059be756c6c461f3a225de25aabda7d5f84a | statute IDs: 100
train {'sha256': 'f3601a7fce994616daf99fec251ca7936e98675a8fbf8af20a782cd55054c8a7', 'cases': 42750, 'tokens': 73374843, 'median_tokens': 648.0, 'max_tokens': 143345}
dev {'sha256': 'ab2660ac23abf38e902d63ab6871df740dc38e3fd973f441745f42564dadbd15', 'cases': 10181, 'tokens': 19509380, 'median_tokens': 813.0, 'max_tokens': 117951}
test {'sha256': 'c699d50bcc2f3e369955268d6195a80b23322ed91cb1fadc8d58b26ca8e9aa77', 'cases': 13019, 'tokens': 24260413, 'median_tokens': 801.0, 'max_tokens': 224282}


## Train partial and distill

In [9]:
import threading
import tempfile
from p5_data import write_json

RUN = Path(tempfile.mkdtemp(prefix="run_", dir=OUT))
write_json(RUN/"run_config.json", dict(config, run_dir=str(RUN)))
print("This run’s outputs:", RUN)

end = SESSION_START + HOURS*3600
jobs_end = end-45
def launch(method, gpu, deadline):
    cmd = [sys.executable,"-u",str(SRC/"p5_train.py"),str(RUN/"run_config.json"),
           method,f"cuda:{gpu}",str(deadline-10)]
    logdir = RUN/method
    logdir.mkdir(exist_ok=True)
    with (logdir/"worker.log").open("w") as file, subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        def relay():
            for line in proc.stdout:
                file.write(line)
                file.flush()
                print(line.rstrip(),flush=True)
        reader = threading.Thread(target=relay,daemon=True)
        reader.start()
        code = proc.wait()
        reader.join()
        if code:
            raise RuntimeError(f"{method} exited with code {code}. See {logdir/'worker.log'}")

ngpu = torch.cuda.device_count()
midpoint = time.time() + (jobs_end-time.time())*.50
launch(METHODS[0],0,midpoint)
launch(METHODS[1],min(1,ngpu-1),jobs_end)

This run’s outputs: /kaggle/working/phase5_A3/run_43pogpgq
partial: 42750 train, 23200 final cases, 0.070s/case, 4 final windows, reserve 45.4 min
partial step 1/42752, loss 0.3365
partial step 16, calibration subset macro-F1 0.0702, eval 12.7s
partial step 25/42752, loss 0.2206
partial step 50/42752, loss 0.2410
partial step 75/42752, loss 0.3758
partial step 100/42752, loss 0.2684
partial step 125/42752, loss 0.6203
partial step 150/42752, loss 0.3943
partial step 160, calibration subset macro-F1 0.1445, eval 12.8s
partial step 175/42752, loss 0.3252
partial step 200/42752, loss 0.5661
partial step 225/42752, loss 0.2296
partial step 250/42752, loss 0.4230
partial step 275/42752, loss 0.4956
partial step 300/42752, loss 0.5897
partial step 320, calibration subset macro-F1 0.1671, eval 12.6s
partial step 325/42752, loss 0.2704
partial step 350/42752, loss 0.5164
partial step 375/42752, loss 0.1843
partial step 400/42752, loss 0.5753
partial step 425/42752, loss 0.3888
partial step 450

## Compare and export

In [10]:
from pathlib import Path
import json
import zipfile
from IPython.display import display, FileLink
import pandas as pd
from p5_metrics import validate
from p5_data import write_json

rows = []
for method in METHODS:
    folder = RUN/method
    r = json.loads((folder/"report.json").read_text())
    test_ids = json.loads((OUT/"test.json").read_text())["ids"]
    with zipfile.ZipFile(folder/"submission.zip") as z:
        assert z.namelist()==["prediction.json"]
        validate(json.loads(z.read("prediction.json")),test_ids,set(manifest["labels"]))
    rows.append(dict(method=method,**r["metrics"],best_step=r["best_step"],total_M=round(r["total_params"]/1e6,2),
                     trainable_M=round(r["trainable_params"]/1e6,2),peak_GiB=r["peak_allocated_GiB"],
                     seconds_per_case=r["inference_seconds_per_case"],prediction_sha256=r["prediction_sha256"]))
table = pd.DataFrame(rows).sort_values(["macro_f1","micro_f1"],ascending=False)
display(table)
table.to_csv(RUN/"comparison.csv",index=False)
winner = table.iloc[0]["method"]
shutil.copyfile(RUN/winner/"submission.zip",RUN/"submission.zip")
decision = {"selected":winner,"selection_criterion":"Held-out dev Macro-F1, then Micro-F1",
            "distinct_predictions":int(table.prediction_sha256.nunique())}
if "distill" in METHODS:
    teacher, student = rows
    decision["student_minus_teacher_macro_f1"] = student["macro_f1"]-teacher["macro_f1"]
    decision["deployed_parameter_reduction_percent"] = 100*(1-student["total_M"]/teacher["total_M"])
    decision["inference_speed_ratio_teacher_over_student"] = teacher["seconds_per_case"]/student["seconds_per_case"]
write_json(RUN/"selection.json",decision)
print(decision)
os.chdir("/kaggle/working")
for path in [RUN/"submission.zip",*(RUN/m/"submission.zip" for m in METHODS)]:
    display(FileLink(str(path.relative_to(Path.cwd()))))

reports = [RUN/m/"report.json" for m in METHODS]
reports += sorted(Path("/kaggle/input").glob("**/phase5_A[1-4]/**/report.json"))
inventory = []
for path in reports:
    r = json.loads(path.read_text())
    if r.get("phase") == 5 and r.get("prediction_sha256"):
        inventory.append(dict(architecture=r["architecture"],method=r["method"],heldout_macro_f1=r["metrics"]["macro_f1"],
                              predictions=r["prediction_sha256"],dataset=r["dataset"],archive=str(path.with_name("submission.zip"))))
index = pd.DataFrame(inventory).sort_values("heldout_macro_f1",ascending=False).drop_duplicates(["architecture","method","predictions"])
index.to_csv(RUN/"submission_inventory.csv",index=False)
display(index[["architecture","method","heldout_macro_f1","predictions"]])
print(f"{index.predictions.nunique()} distinct prediction files visible here. The Phase 5 minimum is six distinct valid submissions.")

,method,macro_f1,micro_f1,subset_accuracy,macro_precision,macro_recall,mean_predictions,best_step,total_M,trainable_M,peak_GiB,seconds_per_case,prediction_sha256
0,partial,0.310917,0.407695,0.022593,0.294945,0.371967,5.714735,7520,266.05,78.27,2.208706,0.058205,c69a60db6791d72358d2551b21ed8c8816f56e5dd5c3cd...
1,distill,0.287991,0.373702,0.012967,0.266013,0.384154,7.026523,8000,150.38,150.38,2.855028,0.027875,fb93f8836db26ef09d174e36903d76e9b77ffc643498df...


{'selected': 'partial', 'selection_criterion': 'Held-out dev Macro-F1, then Micro-F1', 'distinct_predictions': 2, 'student_minus_teacher_macro_f1': -0.022925485490141728, 'deployed_parameter_reduction_percent': 43.47679007705318, 'inference_speed_ratio_teacher_over_student': 2.0880734877432356}


/kaggle/working/phase5_A3/run_43pogpgq/submission.zip

/kaggle/working/phase5_A3/run_43pogpgq/partial/submission.zip

/kaggle/working/phase5_A3/run_43pogpgq/distill/submission.zip

,architecture,method,heldout_macro_f1,predictions
0,A3,partial,0.310917,c69a60db6791d72358d2551b21ed8c8816f56e5dd5c3cd...
1,A3,distill,0.287991,fb93f8836db26ef09d174e36903d76e9b77ffc643498df...


2 distinct prediction files visible here. The Phase 5 minimum is six distinct valid submissions.
